In [ ]:
#python -m pip install --upgrade pip
#pip install requests
#pip install beautifulsoup4
#pip install lxml

In [ ]:
import csv
import os
import time
import re
import requests
from bs4 import BeautifulSoup

## 1. 차트 목록 페이지에서 곡 목록 + 상세페이지 링크(songid) 수집

In [ ]:
url = "https://www.melon.com/chart/index.htm"
hdr = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.melon.com/',
}

res = requests.get(url, headers=hdr)
print(f"상태코드: {res.status_code}")

soup = BeautifulSoup(res.text, "lxml")

song_list_div = soup.find("div", class_="service_list_song")
all_trs = song_list_div.find_all("tr")
song_rows = [tr for tr in all_trs if tr.find("td")]
print(f"수집된 곡 수: {len(song_rows)}")

In [ ]:
# 곡 상세페이지는 songid로 접근: https://www.melon.com/song/detail.htm?songId=XXXXX
# tr 안의 재생 버튼(a 태그)이 보통 onclick="playSong('...', 곡ID)" 형태로 songid를 담고 있음

def extract_songid(tr):
    a_tag = tr.find("a", attrs={"href": re.compile("javascript:melon.link.goSongDetail")})
    if a_tag is None:
        return None
    onclick = a_tag.get("href", "")
    m = re.search(r"goSongDetail\('(\d+)'\)", onclick)
    return m.group(1) if m else None

songs = []
for tr in song_rows:
    rank_tag = tr.find("span", class_="rank")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    title_tag = tr.find("div", class_="ellipsis rank01")
    title = None
    if title_tag:
        a_tag = title_tag.find("a")
        title = a_tag.get_text(strip=True) if a_tag else title_tag.get_text(strip=True)

    artist_tag = tr.find("div", class_="ellipsis rank02")
    artist = None
    if artist_tag:
        a_tags = artist_tag.find_all("a")
        if a_tags:
            artist = a_tags[0].get_text(strip=True)

    songid = extract_songid(tr)

    songs.append({
        "rank": rank,
        "title": title,
        "artist": artist,
        "songid": songid,
    })

print(f"songid 추출된 곡 수: {sum(1 for s in songs if s['songid'])} / {len(songs)}")
print(songs[:3])

## 2. songid로 상세페이지 들어가서 가사/장르/발매일 등 추가 수집

(연습용으로 상위 10곡만 진행 — 전체를 돌리면 요청이 100번 나가므로 서버 부담과 시간 고려)

In [ ]:
def get_song_detail(songid):
    detail_url = f"https://www.melon.com/song/detail.htm?songId={songid}"
    res = requests.get(detail_url, headers=hdr)
    time.sleep(1)  # 매 요청 사이 대기 (서버 부담 줄이기)

    if res.status_code != 200:
        return None

    soup = BeautifulSoup(res.text, "lxml")

    # 장르/발매일 등은 dl.list 안의 dt/dd 쌍으로 들어있음
    genre = None
    release_date = None
    dl = soup.find("dl", class_="list")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        info = {dt.get_text(strip=True): dd.get_text(strip=True) for dt, dd in zip(dts, dds)}
        genre = info.get("장르")
        release_date = info.get("발매일")

    # 가사
    lyric_tag = soup.find("div", id="d_video_summary")
    lyric = lyric_tag.get_text(separator="\n", strip=True) if lyric_tag else None

    return {
        "genre": genre,
        "release_date": release_date,
        "lyric": lyric,
    }


# 연습: 상위 10곡만 상세정보 추가 수집
target_songs = [s for s in songs if s["songid"]][:10]

for s in target_songs:
    detail = get_song_detail(s["songid"])
    if detail:
        s.update(detail)
    print(s["rank"], s["title"], s["artist"], "| 장르:", s.get("genre"), "| 발매일:", s.get("release_date"))

## 3. CSV로 저장

In [ ]:
filename = "멜론_차트_상세.csv"
f = open(filename, "w", encoding="utf-8-sig", newline="")
writer = csv.writer(f)
attributes = ["순위", "곡명", "가수", "장르", "발매일", "가사"]
writer.writerow(attributes)

for s in target_songs:
    writer.writerow([
        s.get("rank"),
        s.get("title"),
        s.get("artist"),
        s.get("genre"),
        s.get("release_date"),
        s.get("lyric"),
    ])

f.close()
print(f"완료: {filename} 저장됨")